# Prompt Engineering Basics

## What is Prompt Engineering?

**Prompt engineering** is the practice of designing, refining, and optimizing inputs (prompts) to large language models (LLMs) to elicit desired outputs reliably and efficiently.

An LLM can be thought of as a **probabilistic function**:

$$f_{LLM}: \text{prompt} \rightarrow P(\text{output} \mid \text{prompt})$$

The model samples from this distribution. Prompt engineering is the art of shaping the input so the most probable outputs are the ones you want.

---

## Anatomy of a Prompt

A well-structured prompt has four components:

| Component | Description | Example |
|-----------|-------------|--------|
| **Instruction** | What task to perform | "Summarize the following text" |
| **Context** | Background information | "You are an expert data scientist" |
| **Input Data** | The actual content to process | "Text: ..." |
| **Output Format** | How the answer should look | "Return JSON with keys: summary, keywords" |

---

## Roles in Chat Models

Modern LLMs use a **chat format** with three roles:

- **System**: Sets the assistant's persona, behavior, and constraints (persistent across turns)
- **User**: The human's input/question
- **Assistant**: The model's response (can be pre-filled to steer output)

```
System:    "You are a helpful Python tutor. Explain concepts simply."
User:      "What is a list comprehension?"
Assistant: "A list comprehension is..."
```

---

## Sampling Parameters

### Temperature
Controls randomness. Modifies the softmax distribution:

$$P(w_i) = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$

- $T \to 0$: deterministic (always picks highest logit)
- $T = 1$: standard sampling
- $T > 1$: more random/creative

### Top-p (Nucleus Sampling)
Sample from the smallest set of tokens whose cumulative probability exceeds $p$:

$$\sum_{w \in V^{(p)}} P(w) \geq p$$

### Top-k
Sample only from the $k$ most probable tokens.

### Max Tokens
Hard limit on response length. Crucial for cost and latency control.

---

## Context Windows

| Model | Context Window |
|-------|---------------|
| GPT-4o | 128K tokens |
| Claude 3.5 Sonnet | 200K tokens |
| Gemini 1.5 Pro | 1M tokens |
| Llama 3.1 70B | 128K tokens |

**Rule of thumb**: 1 token ≈ 0.75 words in English.

---

## Zero-Shot Prompting

Asking the model to perform a task with no examples:

```
Classify the sentiment of this review as Positive, Negative, or Neutral:
Review: "The food was amazing but the service was slow."
Sentiment:
```

---

## Prompt Formatting

### Markdown
Use headers, bold, lists to structure complex prompts.

### XML Tags (Claude's preference)
```xml
<document>
  <content>...text here...</content>
</document>
```

### Delimiters
Use `"""`, `###`, `---`, or `<tags>` to separate sections and prevent injection.

---

## Output Formatting

Always specify the output format explicitly:

```
Return your answer as a JSON object with these exact keys:
{
  "sentiment": "positive|negative|neutral",
  "confidence": 0.0-1.0,
  "reasoning": "brief explanation"
}
```

---

## Common Pitfalls

| Pitfall | Bad | Good |
|---------|-----|------|
| Ambiguity | "Write something about ML" | "Write a 200-word intro to supervised learning for beginners" |
| Underspecification | "Summarize this" | "Summarize in 3 bullet points, each under 20 words" |
| No format | "List ML algorithms" | "List 5 ML algorithms as a markdown table with Name, Type, Use Case" |
| Contradiction | "Be brief and comprehensive" | Pick one or define scope clearly |

In [1]:
# Install: pip install openai anthropic
import os

# ── OpenAI Example ──────────────────────────────────────────────────────────
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def chat(system: str, user: str, temperature: float = 0.7, max_tokens: int = 512):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": user},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content

# Zero-shot example
result = chat(
    system="You are a helpful assistant.",
    user="Classify the sentiment: 'The food was amazing but service was slow.' Return JSON: {sentiment, confidence}"
)
print(result)

In [2]:
import json

# ── Structured Output with JSON mode ────────────────────────────────────────
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{
        "role": "user",
        "content": "Extract: name, age, city from: 'Alice is 30 and lives in Paris'"
    }],
    response_format={"type": "json_object"},
)
data = json.loads(response.choices[0].message.content)
print(data)

In [3]:
# ── Temperature Effect Demo ──────────────────────────────────────────────────
prompt = "Complete this sentence creatively: 'The robot opened its eyes and'"

for temp in [0.0, 0.5, 1.0, 1.5]:
    result = chat("You are creative.", prompt, temperature=temp, max_tokens=30)
    print(f"T={temp}: {result.strip()}")

In [4]:
# ── Anthropic Claude Example ─────────────────────────────────────────────────
import anthropic

claude = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

message = claude.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=256,
    system="You are an expert data scientist. Use XML tags in your response.",
    messages=[{
        "role": "user",
        "content": """
        <task>Explain overfitting in 2 sentences</task>
        <format>Return <explanation> and <analogy> tags</format>
        """
    }]
)
print(message.content[0].text)

## Additional Learning Resources

### Guides & Documentation
- [Prompt Engineering Guide](https://www.promptingguide.ai/) Comprehensive techniques reference
- [OpenAI Prompt Engineering Guide](https://platform.openai.com/docs/guides/prompt-engineering) Official OpenAI guide
- [Anthropic Prompt Library](https://docs.anthropic.com/en/prompt-library/library) Claude prompt examples
- [Google Prompt Design Guide](https://cloud.google.com/vertex-ai/generative-ai/docs/learn/prompts/introduction-prompt-design)

### Papers
- [Pre-train, Prompt, and Predict (Liu et al., 2021)](https://arxiv.org/abs/2107.13586) Survey of prompting methods
- [Large Language Models are Zero-Shot Reasoners (Kojima et al., 2022)](https://arxiv.org/abs/2205.11916)

### Courses
- [DeepLearning.AI ChatGPT Prompt Engineering for Developers](https://www.deeplearning.ai/short-courses/chatgpt-prompt-engineering-for-developers/)
- [Learn Prompting (open source)](https://learnprompting.org/)